# Day 15 · 讓人類插一腳：Human Input 與 Tool Confirmation

> 第三部・戰術編排　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 15 - 讓人類插一腳：Human Input 與 Tool Confirmation.md`

## 今天要學會

1. 用 **Tool Confirmation** 讓危險操作需要人工核可
2. 在 notebook 裡完成「攔截 → 人工決定 → 恢復執行」的完整迴圈
3. 分辨兩套 HITL 系統的適用場合

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. 兩套 HITL，不要搞混

ADK 有**兩套**「讓人類插手」的機制，解決的問題不同：

| | Tool Confirmation | `RequestInput` |
|---|---|---|
| 掛在 | **單一工具** | **圖的節點**（Day 13） |
| 問的是 | 「這個動作可以執行嗎？」 | 「請提供這份資料」 |
| 回答 | 核准 / 拒絕 | 任意結構化資料 |
| 適合 | 刪除、付款、寄信 | 補件、選項確認、審稿 |

原文說的「兩套 HITL 的誤解」就是指這個——很多人以為是同一件事。
本日主要示範 Tool Confirmation，因為它在 notebook 裡可以跑完整流程。

## 2. `require_confirmation=True`

只要在 `FunctionTool` 上加一個參數，這個工具就變成「需要核可」。

In [2]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import FunctionTool
from google.genai import types

EXECUTED: list[str] = []


def delete_account(user_id: str) -> dict:
    """永久刪除一個使用者帳號。這個動作無法復原。

    Args:
        user_id: 使用者編號。
    """
    EXECUTED.append(user_id)
    return {"deleted": user_id, "status": "ok"}


def query_account(user_id: str) -> dict:
    """查詢使用者帳號資訊（唯讀，安全）。

    Args:
        user_id: 使用者編號。
    """
    return {"user_id": user_id, "status": "active", "created": "2024-03-01"}


admin = LlmAgent(
    name="admin",
    model=get_model(),
    instruction=(
        "你是系統管理助理。查詢用 query_account，刪除用 delete_account。"
        "用繁體中文簡短回答。"
    ),
    tools=[
        query_account,                                             # 唯讀，不需核可
        FunctionTool(func=delete_account, require_confirmation=True),  # 危險，需要核可
    ],
)

import inspect

print("FunctionTool 的 require_confirmation 參數:")
print(" ", inspect.signature(FunctionTool.__init__))

FunctionTool 的 require_confirmation 參數:
  (self, func: 'Callable[..., Any]', *, require_confirmation: 'Union[bool, Callable[..., bool]]' = False)


### 先看安全的工具：正常執行

In [3]:
print(await run_once(admin, "查一下帳號 u-100 的狀態", trace=True))

  🔧 [admin] 呼叫 query_account({'user_id': 'u-100'})
  ↩️  [admin] query_account 回傳 {'user_id': 'u-100', 'status': 'active', 'created': '2024-03-01'}


  💬 [admin] 帳號 u-100 的狀態為：**正常啟用 (active)**，建立日期為 2024-03-01。
帳號 u-100 的狀態為：**正常啟用 (active)**，建立日期為 2024-03-01。


## 3. 危險工具：被攔下來了

In [4]:
runner = InMemoryRunner(agent=admin, app_name="day15")
sid = await new_session(runner)
msg = types.Content(role="user", parts=[types.Part(text="請刪除帳號 u-999")])

pending = None
async for ev in runner.run_async(user_id="student", session_id=sid, new_message=msg):
    for p in (ev.content.parts if ev.content else []) or []:
        fc = getattr(p, "function_call", None)
        fr = getattr(p, "function_response", None)
        if fc:
            print(f"  🔧 function_call: {fc.name}")
            if fc.name == "adk_request_confirmation":
                pending = fc
        elif fr:
            print(f"  ↩️  function_response: {str(fr.response)[:110]}")
        elif getattr(p, "text", None):
            print(f"  💬 {p.text.strip()[:90]}")

print(f"\n工具實際執行了嗎？ {EXECUTED}")

  🔧 function_call: delete_account
  🔧 function_call: adk_request_confirmation
  ↩️  function_response: {'error': 'This tool call requires confirmation, please approve or reject.'}

工具實際執行了嗎？ []


三件事發生了：

1. 模型正常發出 `delete_account` 的呼叫
2. **ADK 把它攔下來**，改成發出一個 `adk_request_confirmation`
3. **`delete_account` 完全沒有被執行**（`EXECUTED` 是空的）

這比在 instruction 裡寫「刪除前要先問使用者」可靠得多——
**它是攔在程式層，不是拜託模型自律。**

### 攔截請求裡有什麼

In [5]:
import json

if pending:
    args = dict(pending.args or {})
    print("confirmation 請求的 id:", pending.id)
    print("\n內容：")
    print(json.dumps(args, indent=2, ensure_ascii=False)[:700])

confirmation 請求的 id: adk-53b5d937-c4a4-4984-ba68-ffd9b36ab5fd

內容：
{
  "originalFunctionCall": {
    "id": "call_2462431",
    "args": {
      "user_id": "u-999"
    },
    "name": "delete_account"
  },
  "toolConfirmation": {
    "hint": "Please approve or reject the tool call delete_account() by responding with a FunctionResponse with an expected ToolConfirmation payload.",
    "confirmed": false
  }
}


重點是兩個欄位：

| 欄位 | 內容 |
|---|---|
| `originalFunctionCall` | **原本要執行什麼**（名稱 + 參數）——給人看的依據 |
| `toolConfirmation.confirmed` | 目前是 `False`；你要把它改成 `True` 才會放行 |

你的 UI 就是拿 `originalFunctionCall` 去渲染一個
「即將刪除帳號 u-999，確定嗎？」的對話框。

## 4. 核准：把執行恢復

送回一個 `FunctionResponse`，**`id` 要跟攔截請求的 `id` 一致**。

In [6]:
approve = types.Content(role="user", parts=[types.Part(
    function_response=types.FunctionResponse(
        id=pending.id,                       # ← 一定要對上
        name="adk_request_confirmation",
        response={"confirmed": True},        # ← 核准
    ))])

async for ev in runner.run_async(user_id="student", session_id=sid, new_message=approve):
    for p in (ev.content.parts if ev.content else []) or []:
        if getattr(p, "function_response", None):
            print(f"  ↩️  {str(p.function_response.response)[:110]}")
        elif getattr(p, "text", None):
            print(f"  💬 {p.text.strip()[:90]}")

print(f"\n核准之後，工具執行了嗎？ {EXECUTED}")

  ↩️  {'deleted': 'u-999', 'status': 'ok'}


  💬 帳號 u-999 已成功刪除。

核准之後，工具執行了嗎？ ['u-999']


完整的迴圈跑完了：

```
  模型想執行 → ADK 攔截 → 人類決定 → 送回 FunctionResponse → 真正執行
```

## 5. 拒絕：把它擋掉

In [7]:
EXECUTED.clear()
sid2 = await new_session(runner)
msg2 = types.Content(role="user", parts=[types.Part(text="刪除帳號 u-888")])

pending2 = None
async for ev in runner.run_async(user_id="student", session_id=sid2, new_message=msg2):
    for p in (ev.content.parts if ev.content else []) or []:
        fc = getattr(p, "function_call", None)
        if fc and fc.name == "adk_request_confirmation":
            pending2 = fc

reject = types.Content(role="user", parts=[types.Part(
    function_response=types.FunctionResponse(
        id=pending2.id, name="adk_request_confirmation",
        response={"confirmed": False},       # ← 拒絕
    ))])

async for ev in runner.run_async(user_id="student", session_id=sid2, new_message=reject):
    for p in (ev.content.parts if ev.content else []) or []:
        if getattr(p, "function_response", None):
            print(f"  ↩️  {str(p.function_response.response)[:110]}")
        elif getattr(p, "text", None):
            print(f"  💬 {p.text.strip()[:110]}")

print(f"\n拒絕之後，工具執行了嗎？ {EXECUTED}")

  ↩️  {'error': 'This tool call is rejected.'}


  💬 抱歉，刪除帳號的操作被拒絕了。

拒絕之後，工具執行了嗎？ []


## 6. 📌 `id` 對不上會怎樣（實測結果跟傳言不同）

原文提到一個 **`invocation_id` 陷阱**：回覆沒對上 id 時會「靜默地開一個新輪次」，
你以為核准了、其實沒有。

**在 ADK 2.8 上實測，結果不是這樣**——它會直接拋例外。實際跑一次看看：

In [8]:
EXECUTED.clear()
sid3 = await new_session(runner)
msg3 = types.Content(role="user", parts=[types.Part(text="刪除帳號 u-777")])

pending3 = None
async for ev in runner.run_async(user_id="student", session_id=sid3, new_message=msg3):
    for p in (ev.content.parts if ev.content else []) or []:
        fc = getattr(p, "function_call", None)
        if fc and fc.name == "adk_request_confirmation":
            pending3 = fc

# 故意用一個錯的 id
wrong = types.Content(role="user", parts=[types.Part(
    function_response=types.FunctionResponse(
        id="adk-this-id-does-not-exist",
        name="adk_request_confirmation",
        response={"confirmed": True},
    ))])

errored = False
try:
    async for ev in runner.run_async(user_id="student", session_id=sid3, new_message=wrong):
        for p in (ev.content.parts if ev.content else []) or []:
            if getattr(p, "text", None):
                print(f"  💬 {p.text.strip()[:100]}")
except Exception as exc:
    errored = True
    print(f"  拋出例外: {type(exc).__name__}: {str(exc)[:120]}")

print(f"\n有報錯嗎？ {'✅ 有' if errored else '❌ 沒有'}")
print(f"工具執行了嗎？ {EXECUTED}")

  拋出例外: ValueError: Function call not found for function response ids: {'adk-this-id-does-not-exist'}. Ensure each function response ID matc

有報錯嗎？ ✅ 有
工具執行了嗎？ []


### 好消息：ADK 2.8 會替你把關

它拋出的是：

```
ValueError: Function call not found for function response ids:
            {'adk-this-id-does-not-exist'}.
            Ensure each function response ID matches ...
```

這比「靜默失敗」好太多了——**錯了你會立刻知道**。

> **給讀舊資料的人**：網路上（含部分中文教學）會說這裡會靜默開新輪次。
> 那可能是舊版行為或誤傳。以你自己環境跑出來的結果為準——
> 這也是本教材每個結論都附上可執行 cell 的原因。

### 但還是有你要自己防的部分

ADK 驗證的是「**這個 id 存不存在於這個 session**」。它**不會**替你檢查：

| 風險 | ADK 會擋嗎 |
|---|---|
| id 根本不存在 | ✅ 會 |
| **重放**：同一個核准送兩次 | ❌ 不會 |
| **越權**：A 使用者核准 B 的請求 | ❌ 不會（它不知道你的權限模型） |
| 核准過期（隔了三天才回） | ❌ 不會 |

### 實務上怎麼防

```python
# 1. 攔截時把 id 存下來（連同誰、什麼時候、要做什麼）
pending_store[confirmation.id] = {
    "session_id": sid,
    "requested_by": current_user,                    # ← 誰發起的
    "call": dict(confirmation.args["originalFunctionCall"]),
    "requested_at": now(),
}

# 2. 收到人工回覆時做 ADK 不會做的三件檢查
rec = pending_store.get(reply_id)
if rec is None:
    raise ValueError("未知或已處理的確認")           # 防重放
if rec["requested_by"] != approver_scope:
    raise PermissionError("無權核准這筆請求")        # 防越權
if now() - rec["requested_at"] > timedelta(hours=1):
    raise TimeoutError("核准已逾期")                 # 防過期

# 3. 處理完立刻移除
del pending_store[reply_id]
```

**權限、重放、過期這三件事 ADK 不會替你做**，因為它不知道你的權限模型。

## 7. 動態決定要不要確認

`require_confirmation` 也接受一個**函式**——可以依參數決定。
例如「小額自動放行、大額才要簽核」。

In [9]:
print("require_confirmation 的型別:",
      inspect.signature(FunctionTool.__init__).parameters["require_confirmation"].annotation)

TRANSFERRED: list[int] = []


def transfer_money(to_account: str, amount: int) -> dict:
    """匯款到指定帳戶。

    Args:
        to_account: 收款帳號。
        amount: 金額（新台幣）。
    """
    TRANSFERRED.append(amount)
    return {"to": to_account, "amount": amount, "status": "completed"}


def needs_approval(**kwargs) -> bool:
    """一萬元以上才需要人工核可——政策寫在程式裡，不是 prompt 裡。"""
    return int(kwargs.get("amount", 0)) >= 10000


bank = LlmAgent(
    name="bank",
    model=get_model(),
    instruction="使用者要匯款時呼叫 transfer_money。用繁體中文簡短回答。",
    tools=[FunctionTool(func=transfer_money, require_confirmation=needs_approval)],
)

require_confirmation 的型別: Union[bool, Callable[..., bool]]


In [10]:
async def try_transfer(text: str) -> None:
    r = InMemoryRunner(agent=bank, app_name="day15b")
    s = await new_session(r)
    m = types.Content(role="user", parts=[types.Part(text=text)])
    needed = False
    async for ev in r.run_async(user_id="student", session_id=s, new_message=m):
        for p in (ev.content.parts if ev.content else []) or []:
            fc = getattr(p, "function_call", None)
            if fc and fc.name == "adk_request_confirmation":
                needed = True
    print(f"  {'⏸ 需要核可' if needed else '✅ 直接放行'}")


TRANSFERRED.clear()
print("匯款 3,000 元：")
await try_transfer("幫我匯 3000 元到帳號 A-1")
print(f"  已完成的匯款: {TRANSFERRED}")

print("\n匯款 50,000 元：")
await try_transfer("幫我匯 50000 元到帳號 A-1")
print(f"  已完成的匯款: {TRANSFERRED}")

匯款 3,000 元：


  ✅ 直接放行
  已完成的匯款: [3000]

匯款 50,000 元：


  ⏸ 需要核可
  已完成的匯款: [3000]


**政策寫在 `needs_approval()` 這個 Python 函式裡**，
不是寫在 instruction 裡拜託模型判斷。可以寫單元測試，可以稽核。

## 8. 另一套：`RequestInput`（圖節點用）

當你要的不是「核准 / 拒絕」，而是**請人補一份資料**時，
用的是 Day 13 那張圖裡的 `RequestInput`。

In [11]:
from google.adk.events import RequestInput

print("RequestInput 的欄位：")
for f, i in RequestInput.model_fields.items():
    req = "必填" if i.default is ... or str(i.default) == "PydanticUndefined" else f"預設 {i.default!r}"
    print(f"  {f:18s} {req}")

RequestInput 的欄位：
  interrupt_id       必填
  payload            預設 None
  message            預設 None
  response_schema    預設 None


| 欄位 | 用途 |
|---|---|
| `interrupt_id` | 這次中斷的識別碼（**跟上面的 `id` 一樣要對上**） |
| `message` | 給人看的說明：「請提供收件地址」 |
| `response_schema` | 期望回什麼結構 |
| `payload` | 附帶的參考資料 |

用法是在 `@node` 裡 `yield RequestInput(...)`，圖就會暫停：

```python
@node
async def collect_address(ctx: Context):
    yield RequestInput(
        interrupt_id="addr-1",
        message="請提供收件地址",
        response_schema=Address,        # Pydantic model
    )
```

### ⚠️ `response_schema` **不會**自動驗證人類的回覆

原文特別提醒這一點：`response_schema` 是給**你的 UI** 看的提示
（要渲染什麼表單欄位），**ADK 不會替你檢查人類真的照格式回覆**。
驗證要自己做：

In [12]:
from pydantic import BaseModel, Field


class Address(BaseModel):
    recipient: str = Field(description="收件人")
    zipcode: str = Field(description="郵遞區號")
    street: str = Field(description="地址")


human_reply = {"recipient": "Sean", "street": "信義路五段 7 號"}   # 少了 zipcode

try:
    Address.model_validate(human_reply)
    print("通過驗證")
except Exception as exc:
    print("❌ 自己驗證才會發現缺欄位：")
    print(str(exc)[:220])

❌ 自己驗證才會發現缺欄位：
1 validation error for Address
zipcode
  Field required [type=missing, input_value={'recipient': 'Sean', 'st...'信義路五段 7 號'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing


## 9. ⚠️ `except BaseException` 會吃掉中斷

這是原文提到的另一個地雷，而且很隱蔽。

ADK 用**例外**來實作「暫停」。如果你在節點裡寫了太寬的 `except`，
就會把暫停訊號吞掉——HITL 直接失效，而且看起來像正常執行完了。

In [13]:
print("""❌ 危險的寫法：

    @node
    async def risky(ctx):
        try:
            ...
            yield RequestInput(...)          # 要暫停
        except BaseException as e:           # ← 把 NodeInterruptedError 也吃掉了
            log(e)
            return {"status": "ok"}          # 假裝成功

✅ 正確的寫法：

    from google.adk.workflow import NodeTimeoutError

    @node
    async def safe(ctx):
        try:
            ...
            yield RequestInput(...)
        except Exception as e:               # ← 只接 Exception，不要 BaseException
            log(e)
            raise                            # 而且該往上拋就往上拋
""")

❌ 危險的寫法：

    @node
    async def risky(ctx):
        try:
            ...
            yield RequestInput(...)          # 要暫停
        except BaseException as e:           # ← 把 NodeInterruptedError 也吃掉了
            log(e)
            return {"status": "ok"}          # 假裝成功

✅ 正確的寫法：

    from google.adk.workflow import NodeTimeoutError

    @node
    async def safe(ctx):
        try:
            ...
            yield RequestInput(...)
        except Exception as e:               # ← 只接 Exception，不要 BaseException
            log(e)
            raise                            # 而且該往上拋就往上拋



**記住**：`BaseException` 涵蓋了 `KeyboardInterrupt`、`SystemExit`、
以及 ADK 用來實作控制流的內部例外。**永遠不要 catch 它。**

## 10. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| `Function call not found for function response ids` | `FunctionResponse` 的 `id` 沒對上（好消息：2.8 會擋） |
| 同一個核准被送兩次都成功 | **ADK 不防重放**，要自己維護 pending_store |
| 危險操作還是被執行了 | 只在 instruction 裡叮嚀，沒用 `require_confirmation` |
| HITL 完全失效、看起來卻正常 | 節點裡 `except BaseException` 把中斷訊號吃掉了 |
| 人類回的資料缺欄位 | `response_schema` **不會自動驗證**，要自己 `model_validate` |
| 同一個確認被重複核准 | 沒有在處理後移除 pending 記錄 |

## 11. 動手練習

1. 把 `needs_approval` 改成「收款帳號不在白名單就要核可」，
   確認政策換一個維度也一樣好寫。
2. 在第 4 節核准後，**再送一次同樣的核准**，觀察 ADK 擋不擋（重放問題）。
3. 寫一個 `pending_store` dict，把第 6 節的驗證補上，
   讓錯誤的 id 會拋出清楚的例外。
4. 把 `delete_account` 也加上 `require_confirmation=needs_approval` 形式的
   動態判斷（例如只有 `u-` 開頭的測試帳號可以直接刪）。

## 本日回顧

- **兩套 HITL 不同**：Tool Confirmation 問「可以執行嗎」，
  `RequestInput` 問「請給我資料」。
- **`require_confirmation=True` 攔在程式層**，比在 instruction 裡叮嚀可靠得多；
  它也接受**函式**，讓政策依參數動態決定。
- **完整迴圈**：攔截 → 拿 `originalFunctionCall` 給人看 →
  回送 `FunctionResponse(id=..., response={"confirmed": True})` → 執行。
- **📌 `id` 對不上會直接拋 `ValueError`**（實測；跟部分舊資料說的「靜默失敗」不同）。
  但 ADK **不防重放、不管權限、不管過期**——這三件要自己維護 pending_store。
- **⚠️ `response_schema` 不會自動驗證人類的回覆**，要自己 `model_validate`。
- **⚠️ 永遠不要 `except BaseException`**——會吃掉 ADK 的中斷訊號。

---
**下一天 → `../day16_template_workflow_agents/`**